<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025s1/blob/main/week13_Transformer/GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Transformer-like (both encoder and decoder) model from scratch for Text Classification

with customized attention block

At the end of this session, you will be able to:
- prepare data and use model-specific Tokenizer to format data suitable for use by the model
- construct the transformer model 
- train the model for binary and multi-class text classification


### Install Hugging Face Transformers library

If you are running this notebook in Google Colab, you will need to install the Hugging Face transformers library as it is not part of the standard environment.

In [ ]:
!pip install transformers

In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split

## Data Preparation

In [ ]:
test_data_url = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/datasets/imdb_test.csv'
train_data_url = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/datasets/imdb_train.csv'

In [ ]:
train_df = pd.read_csv(train_data_url)
test_df = pd.read_csv(test_data_url)

In [ ]:
train_df.head()

The train set has 40000 samples. We will use only a small subset (e.g. 2000) samples for finetuning our pretrained model. Similarly we will use a smaller test set for evaluating our model.  We use dataframe's `sample()` to randomly select a subset of samples.

In [ ]:
TRAIN_SIZE = 2000
TEST_SIZE = 200 

train_df = train_df.sample(n=TRAIN_SIZE)
test_df = test_df.sample(n=TEST_SIZE)

We now convert the text label into numeric values of 0 (negative) and 1 (positive) 

In [ ]:
train_df['sentiment'] =  train_df['sentiment'].apply(lambda x: 0 if x == 'negative' else 1)
test_df['sentiment'] =  test_df['sentiment'].apply(lambda x: 0 if x == 'negative' else 1)

In [ ]:
train_df.sentiment.value_counts()

In [ ]:
train_texts = train_df['review']
train_labels = train_df['sentiment']
test_texts = test_df['review']
test_labels = test_df['sentiment']

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(train_texts, train_labels, test_size=.2)

## Tokenization

We will now load the DistilBert tokenizer for the pretrained model "distillbert-base-uncased".  The tokenizer helps to produce the input tokens that are suitable to be used by the DistilBert model, e.g. it automatically append the \[CLS\] token in the front of the sequence of tokens and the \[SEP\] token at the end of the sequence of tokens , and also the attention mask for those padded positions in the input sequence of tokens.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

The DistilBERT tokenizer (identical to Bert tokenizer) use WordPiece vocabulary. It has close to 30000 words. Each word has its own ids, we would need to map the tokens to those ids.

In [ ]:
print(f"Tokenizer vocab size = {tokenizer.vocab_size}")
print(list(tokenizer.vocab.keys())[6000:6020])

Let us take a closer look at the output of the tokenization process. 

We notice that the tokenizer will return a dictionary of two items 'input_ids' and 'attention_mask'. The input_ids contains the IDs of the tokens. While the 'attention_mask' contains the masking pattern for those padded positions. If you are using BERT tokenizer, there will be additional item called 'token_type_ids'.

We also notice that for the example sentence, the word 'Transformer' is being broken up into two tokens 'Trans' and '##former'. The '##' means that the rest of the token should be attached to the previous one.

We also see that the tokenizer appended \[CLS\] to the beginning of the token sequence, and \[SEP\] at the end. 

In [ ]:
test_sentence = "Transformer is really good for Natural Language Processing."

encoding = tokenizer(test_sentence, padding=True, truncation=True)
print(f"Encoding keys:  {encoding.keys()}\n")

print(f"token ids: {encoding['input_ids']}\n")
print(f"attention_mask: {encoding['attention_mask']}\n")
print(f"tokens: {tokenizer.convert_ids_to_tokens(encoding['input_ids'])}")

Now let's go ahead and tokenize our texts. But before we do so, we need to convert the pandas series to list first as the tokenizer cannot work with pandas series or dataframe directly. 

In [ ]:
train_texts = train_texts.to_list()
train_labels = train_labels.to_list()
val_texts = val_texts.to_list()
val_labels = val_labels.to_list()
test_texts = test_texts.to_list()
test_labels = test_labels.to_list()

In [ ]:
train_encodings = tokenizer(train_texts, padding=True, truncation=True)
val_encodings = tokenizer(val_texts, padding=True, truncation=True)
test_encodings = tokenizer(test_texts, padding=True, truncation=True)

We then create a tensorflow dataset using the encodings and the labels.

In [ ]:
batch_size = 16

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(batch_size)

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_labels
)).batch(batch_size)

## Construct a customized GPT-like model

Now let us fine-tune the pre-trained model by training it with our custom dataset.  

We will instantiate a pretrained model 'distilbert-base-uncased', using `TFAutoModelForSequenceClassification`, and passing `num_labels=2` to indicate we want to train a 2-class (binary) classifier.

The model is a `tf.keras.Model` subclass. So you can train the model using Keras API such as `fit()`.

In [ ]:
import tensorflow as tf


def causal_mask(L, dtype):
    """Return lower-tri (L, L) mask as 0/-1e9 additive bias."""
    m = tf.linalg.band_part(tf.ones((L, L), dtype=dtype), -1, 0)
    return (1.0 - m) * tf.constant(-1e9, dtype)


In [ ]:
class CustomAttention(tf.keras.layers.Layer):
    """
    Multi-head attention with
      • Rotary position embedding (RoPE)
      • Per-head learnable gate α_h
      • Works in 3 modes: 'bi', 'causal', 'cross'
        • 'bi'     → full bidirectional (encoder)
        • 'causal' → lower-tri (decoder self)
        • 'cross'  → query = decoder,  key/value = encoder
    """

    def __init__(self, hidden, heads, dropout=0.1, **kw):
        super().__init__(**kw)
        assert hidden % heads == 0
        self.hid, self.h = hidden, heads
        self.dk = hidden // heads
        self.dropout = dropout

        self.proj_q = tf.keras.layers.Dense(hidden, use_bias=False)
        self.proj_k = tf.keras.layers.Dense(hidden, use_bias=False)
        self.proj_v = tf.keras.layers.Dense(hidden, use_bias=False)
        self.proj_o = tf.keras.layers.Dense(hidden)
        self.alpha   = self.add_weight("alpha", shape=(heads,), initializer="ones")

    # ---- RoPE helper ---- #
    def _rope(self, x):
        # x (B, H, L, dk)  dk even
        dk2 = self.dk // 2
        freq = tf.range(dk2, dtype=x.dtype) / dk2
        freq = 1.0 / (10000.0 ** freq)                    # (dk/2,)
        pos  = tf.range(tf.shape(x)[-2], dtype=x.dtype)   # (L,)
        rot  = tf.einsum("l,f->lf", pos, freq)            # (L, dk/2)
        cos, sin = tf.cos(rot), tf.sin(rot)
        x1, x2 = tf.split(x, 2, axis=-1)
        x_rot = tf.concat([x1*cos - x2*sin, x1*sin + x2*cos], axis=-1)
        return x_rot

    # ---- head reshape ---- #
    def _split(self, t):
        # (B,L,H) -> (B,head,L,dk)
        B, L = tf.shape(t)[0], tf.shape(t)[1]
        t = tf.reshape(t, [B, L, self.h, self.dk])
        return tf.transpose(t, [0, 2, 1, 3])

    # ---- core call ---- #
    def call(self, q_inp, k_inp=None, v_inp=None,
             mask=None, mode="bi", training=False):
        """
        mode:
          • 'bi'     → q_inp==k_inp, bidirectional
          • 'causal' → q_inp==k_inp, causal mask
          • 'cross'  → cross attention (decoder–>encoder)
        """
        if k_inp is None:           # self-attn
            k_inp, v_inp = q_inp, q_inp

        q = self._split(self.proj_q(q_inp))
        k = self._split(self.proj_k(k_inp))
        v = self._split(self.proj_v(v_inp))

        # rotary on q,k when same length (self attn)
        if mode != "cross":
            q, k = self._rope(q), self._rope(k)

        # scaled dot-product
        logits = tf.matmul(q, k, transpose_b=True)      # (B,H,Lq,Lk)
        logits /= tf.math.sqrt(tf.cast(self.dk, logits.dtype))
        logits *= self.alpha[None, :, None, None]       # gate

        # masks ----------------------------------------------------
        if mode == "causal":
            L = tf.shape(q)[-2]
            logits += causal_mask(L, logits.dtype)
        if mask is not None:           # mask = (B, 1, 1, Lk) bool
            logits = tf.where(mask, logits,
                              tf.constant(-1e9, logits.dtype))

        attn = tf.nn.softmax(logits, axis=-1)
        attn = tf.nn.dropout(attn, self.dropout) if training else attn

        out = tf.matmul(attn, v)                       # (B,H,Lq,dk)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, [tf.shape(out)[0], tf.shape(out)[1], self.hid])
        return self.proj_o(out)


In [ ]:
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(self, hidden, heads, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.attn  = CustomAttention(hidden, heads, drop)
        self.mlp   = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(hidden * mlp_ratio, activation="gelu"),
                tf.keras.layers.Dropout(drop),
                tf.keras.layers.Dense(hidden),
                tf.keras.layers.Dropout(drop),
            ]
        )

    def call(self, x, training=False, mask=None):
        x = x + self.attn(self.norm1(x), mask=mask, mode="bi", training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        return x


class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, hidden, heads, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm3 = tf.keras.layers.LayerNormalization(epsilon=1e-5)

        self.self_attn  = CustomAttention(hidden, heads, drop)
        self.cross_attn = CustomAttention(hidden, heads, drop)
        self.mlp = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(hidden * mlp_ratio, activation="gelu"),
                tf.keras.layers.Dropout(drop),
                tf.keras.layers.Dense(hidden),
                tf.keras.layers.Dropout(drop),
            ]
        )

    def call(self, x, enc, training=False, self_mask=None, enc_mask=None):
        x = x + self.self_attn(self.norm1(x), mode="causal",
                               mask=self_mask, training=training)
        x = x + self.cross_attn(self.norm2(x), k_inp=enc, v_inp=enc,
                                mode="cross", mask=enc_mask, training=training)
        x = x + self.mlp(self.norm3(x), training=training)
        return x


In [ ]:
class Transformer(tf.keras.Model):
    """
    Encoder-decoder Transformer with shared vocab embeddings.
    """

    def __init__(
        self,
        vocab,
        max_len,
        num_enc=4,
        num_dec=4,
        hidden=256,
        heads=8,
        mlp_ratio=4,
        drop=0.1,
        **kw,
    ):
        super().__init__(**kw)
        self.hidden = hidden
        self.tok_emb = tf.keras.layers.Embedding(vocab, hidden)
        self.pos_emb = self.add_weight("pos_emb",
                                       shape=(max_len, hidden),
                                       initializer="zeros")
        self.drop = tf.keras.layers.Dropout(drop)

        self.enc_layers = [
            EncoderBlock(hidden, heads, mlp_ratio, drop) for _ in range(num_enc)
        ]
        self.dec_layers = [
            DecoderBlock(hidden, heads, mlp_ratio, drop) for _ in range(num_dec)
        ]
        self.enc_norm = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dec_norm = tf.keras.layers.LayerNormalization(epsilon=1e-5)

    # ---- forward  ---- #
    def call(self, src_ids, tgt_cls_ids, training=False):
        """
        src_ids      : (B, Ls)  input sequence (0 = pad)
        tgt_cls_ids  : (B, 1)   always a single learned [CLS] id (e.g. 1)
        """
        src_mask = tf.not_equal(src_ids, 0)[:, None, None, :]  # (B,1,1,Ls)
        enc = self.tok_emb(src_ids) + self.pos_emb[: tf.shape(src_ids)[1]]
        enc = self.drop(enc, training=training)
        for blk in self.enc_layers:
            enc = blk(enc, training=training, mask=src_mask)
        enc = self.enc_norm(enc)

        # decoder consumes 1-token sequence “[CLS]”
        dec = self.tok_emb(tgt_cls_ids) + self.pos_emb[:1]
        dec = self.drop(dec, training=training)
        for blk in self.dec_layers:
            dec = blk(dec, enc,
                      training=training,
                      self_mask=None, enc_mask=src_mask)
        dec = self.dec_norm(dec)          # (B,1,H)

        return dec[:, 0]                  # pooled vector (B,H)


In [ ]:
class Classifier(tf.keras.Model):
    def __init__(self, transformer: Transformer, n_labels, drop=0.2, **kw):
        super().__init__(**kw)
        self.trans = transformer
        self.drop  = tf.keras.layers.Dropout(drop)
        self.out   = tf.keras.layers.Dense(n_labels, activation="softmax")

    def call(self, src_ids, training=False):
        cls_id = tf.fill([tf.shape(src_ids)[0], 1], 1)   # assume 1 = CLS token
        pooled = self.trans(src_ids, cls_id, training=training)  # (B,H)
        return self.out(self.drop(pooled, training=training))


In [ ]:
VOCAB_SIZE = 30_000
MAX_LEN    = 128
NUM_LABELS = 4

tf.keras.mixed_precision.set_global_policy("mixed_float16")  # optional

transformer = Transformer(
    vocab=VOCAB_SIZE,
    max_len=MAX_LEN,
    num_enc=6,
    num_dec=2,
    hidden=384,
    heads=6,
    mlp_ratio=4,
)

model = Classifier(transformer, NUM_LABELS)

# model.compile(
#     optimizer=tf.keras.optimizers.AdamW(3e-4, weight_decay=1e-2),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"],
# )
# model.summary(line_length=130)


In [ ]:
# from transformers import TFAutoModelForSequenceClassification

# model = TFAutoModelForSequenceClassification.from_pretrained(
#         "distilbert-base-uncased",num_labels=2)

Transformer models benefit from a much lower learning rate than the default used by Adam, which is 1e-3. In this training, we will start the training with 5e-5 (0.00005) and slowly reduce the learning rate over the course of training. In the literature, you will sometimes see this referred to as decaying or annealing the learning rate. In Keras, the best way to do this is to use a learning rate scheduler. A good one to use is PolynomialDecay. Despite the name, with default settings it simply linearly decays the learning rate from the initial value to the final value over the course of training, which is exactly what we want. In order to use a scheduler correctly, though, we need to tell it how long training is going to be. We compute that as `num_train_steps` below.

In [ ]:
from tensorflow.keras.optimizers.schedules import PolynomialDecay

num_epochs = 10

# The number of training steps is the number of samples in the dataset, divided by the batch size then multiplied
# by the total number of epochs. Since our dataset is already batched, we can simply take the len.
num_train_steps = len(train_dataset) * num_epochs

lr_scheduler = PolynomialDecay(
    initial_learning_rate=5e-5, end_learning_rate=0.0, decay_steps=num_train_steps
)

Now we will just compile the model with the learning rate scheduler and the loss function and train our model for 1 epoch. 

Note that the transformer model output logits directly instead of going through a softmax layer. In your loss function, you will need to set `from_logits=True`.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

opt = Adam(learning_rate=lr_scheduler)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])

model.fit(train_dataset, validation_data=val_dataset, epochs=num_epochs)

You will notice that validation accuracy reaches around 89%.  Let's evaluate on our test set. We should see around the same accuracy. 


In [ ]:
model.evaluate(test_dataset)

Let's just go ahead and save our model for inference later. Note that we use transformers library specific save method `save_pretrained()` instead of normal keras model save.

In [ ]:
model.save_pretrained('finetuned_model')

## Try out the model

Now let's try out our model with our own sentence.  We first load our saved fined-tuned model using `from_pretrained()` method and specify the folder name where we saved the model to.

In [ ]:
my_model = TFAutoModelForSequenceClassification.from_pretrained(
        "finetuned_model")

In [ ]:
text = input('Write your review here:')

In [ ]:
inputs = tokenizer(text, return_tensors="tf")
output = my_model(inputs)
pred_prob = tf.nn.softmax(output.logits, axis=-1)
print(pred_prob)
pred = np.argmax(pred_prob)
print(pred)
if pred == 1:
    print('positive')
else:
    print('negative')

**Exercise:**

Now, try to fine-tune DistilBERT for  multi-class text classification task using this [dataset](https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/it3103/news.csv) that groups news title into 4 categories: e (entertainment), b (business), t (tech), m (medical/health). Original dataset can be found [here](https://archive.ics.uci.edu/ml/datasets/News+Aggregator)

*Hint*:

- The csv file is using tab as delimiter, so you need to specify `delimiter='\t'` when you use `pd.read_csv()`
- You should also write a separate function to map the 4 character labels `('e','t','b','m')` into its numeric labels
- Remember to change the `num_labels` to the appropriate number when you instantiate the DistilBert SequenceClassification model.